[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/25_flash_attention.ipynb)

# 🔴 Hard: Flash Attention (Tiled)

Implement **tiled attention with online softmax** — the core idea behind Flash Attention.

### Signature
```python
def flash_attention(Q, K, V, block_size=32) -> Tensor:
    # Q, K, V: (B, S, D)
    # Returns: (B, S, D) — same as standard attention
```

### Key Insight
Instead of materializing the full S×S attention matrix, process in blocks:
1. For each Q-block, iterate over K/V blocks
2. Use **online softmax**: track running `max` and `sum`
3. Rescale accumulator when max changes: `acc *= exp(old_max - new_max)`
4. Final: `output = acc / row_sum`

Must give **identical** results to standard softmax attention.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE

def flash_attention(Q, K, V, block_size=32):
    # Process Q in blocks, iterate K/V blocks with online softmax
    B, S, D = Q.shape
    scale = 1 / math.sqrt(D)
    device = Q.device

    O = torch.zeros_like(Q)

    for i_start in range(0, S, block_size):
        i_end = min(i_start + block_size, S)
        Q_block = Q[:, i_start:i_end]

        running_max = torch.full((B, i_end - i_start), float('-inf'), device=device)
        running_denom = torch.zeros((B, i_end - i_start), device=device)
        running_output = torch.zeros((B, i_end - i_start, D), device=device)

        for j in range(0, S, block_size):
            j_end = min(j + block_size, S)
            K_block = K[:, j:j_end]
            V_block = V[:, j:j_end]

            block_scores = (Q_block @ K_block.transpose(-2, -1)) * scale
            
            block_max = block_scores.max(dim=-1).values
            new_max = torch.maximum(running_max, block_max)

            rescale = torch.exp(running_max - new_max)
            block_weights = torch.exp(block_scores - new_max.unsqueeze(-1))

            running_denom = rescale * running_denom + block_weights.sum(dim=-1)
            running_output = rescale.unsqueeze(-1) * running_output + block_weights @ V_block
            running_max = new_max

        O[:, i_start:i_end] = running_output / running_denom.unsqueeze(-1)
        
    return O
        

In [13]:
# 🧪 Debug
import math
Q, K, V = torch.randn(1, 8, 4), torch.randn(1, 8, 4), torch.randn(1, 8, 4)
out = flash_attention(Q, K, V, block_size=4)
scores = torch.bmm(Q, K.transpose(1,2)) / math.sqrt(4)
ref = torch.bmm(torch.softmax(scores, dim=-1), V)
print('Match:', torch.allclose(out, ref, atol=1e-4))

Match: True


In [14]:
# ✅ SUBMIT
from torch_judge import check
check('flash_attention')


🧪 Testing: Flash Attention (Tiled) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Matches standard attention (3.7ms)
  ✅ [2/4] Non-aligned block size (1.5ms)
  ✅ [3/4] Block size invariant (2.8ms)
  ✅ [4/4] Gradient flow (2.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (10.6ms total)
  Progress saved. Run status() to see your dashboard.

